# 2. Normalization and Highly Variable Genes

The quality-controlled AnnData object generated in Section 1 is prepared for
downstream single-cell RNA-seq analysis.

## Workflow

Final QC-filtered data → preserve counts → normalization → log transformation
→ highly variable gene (HVG) selection → HVG subsetting → regression →
scaling → final dataset for dimensionality reduction.

## Input

`results/preprocessing_tables/merged_qc_filtered.h5ad`

## Main outputs

- `results/preprocessing_tables/normalization_summary.csv`
- `results/figures/08_highly_variable_genes.png`
- `results/preprocessing_tables/hvg_summary.csv`
- `results/preprocessing_tables/top_hvgs.csv`
- `results/preprocessing_tables/hvg_subset.h5ad`
- `results/preprocessing_tables/normalized_hvg_regressed_scaled.h5ad`

> **Reproducibility:** No Google Colab or Google Drive paths are used. The
> project root is detected automatically, so the notebook can be run after
> cloning the repository either from the repository root or from the
> `notebooks/` directory.


## 2.1 Import libraries and locate the project

The notebook searches the current directory and its parent directories for
the project structure. This avoids hard-coded user-specific paths.


In [1]:
from pathlib import Path
from importlib.metadata import version

import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.set_figure_params(
    dpi=100,
    dpi_save=300,
    figsize=(6, 4)
)

def find_project_root():
    current = Path.cwd().resolve()

    candidates = [current, *current.parents]

    for path in candidates:
        if (
            (path / "data").is_dir()
            and (path / "notebooks").is_dir()
            and (path / "results").is_dir()
        ):
            return path

    raise FileNotFoundError(
        "Could not locate the project root. "
        "Run this notebook from inside the sc-project3-IFN-I repository."
    )

project_root = find_project_root()

data_dir = project_root / "data"
results_dir = project_root / "results"
preprocessing_dir = results_dir / "preprocessing_tables"
figures_dir = results_dir / "figures"

preprocessing_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

input_file = preprocessing_dir / "merged_qc_filtered.h5ad"

print("Project root:", project_root)
print("Pandas:", pd.__version__)
print("Scanpy:", sc.__version__)
print("AnnData:", version("anndata"))
print("Input exists:", input_file.exists())


Project root: /home/naimur-neer/Documents/sc-project3-IFN-I/sc-project3-IFN-I
Pandas: 2.3.3
Scanpy: 1.11.5
AnnData: 0.12.19
Input exists: True


/tmp/ipykernel_24469/1906074086.py:49: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("Scanpy:", sc.__version__)


## 2.2 Load the final QC-filtered dataset

This is the output of Section 1. It has already undergone:

- basic cell QC
- Scrublet-predicted doublet removal
- gene filtering for genes detected in at least five cells

This dataset is the starting point for expression preprocessing.


In [2]:
if not input_file.exists():
    raise FileNotFoundError(
        f"Required input file was not found: {input_file}\n"
        "Run notebooks/01_data_loading_qc.ipynb first."
    )

adata = sc.read_h5ad(input_file)

print(adata)
print(f"Cells: {adata.n_obs:,}")
print(f"Genes: {adata.n_vars:,}")


AnnData object with n_obs × n_vars = 87448 × 36601
    obs: 'sample', 'donor_type', 'batch', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'doublet_score', 'predicted_doublet', 'doublet_info'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'donor_type_colors', 'doublet_info_colors', 'sample_colors'
Cells: 87,448
Genes: 36,601


## 2.3 Preserve the raw count matrix

The original count matrix is copied to `adata.layers["counts"]` before any
normalization or transformation.

This preserves the count data for later analyses that require the original
expression counts.


In [3]:
adata.layers["counts"] = adata.X.copy()

print("Count layer created:", "counts" in adata.layers)


Count layer created: True


## 2.4 Normalize and log-transform expression

Total-count normalization scales each cell to a target library size of
10,000 counts. This reduces the effect of differences in sequencing depth
between cells.

The normalized expression values are then log-transformed using `log1p`.

The complete normalized/log-transformed matrix is stored in `adata.raw`
before the AnnData object is subset to HVGs.


In [4]:
# Record sample-level information before normalization
normalization_summary = (
    adata.obs
    .groupby("sample", observed=True)
    .agg(
        cells=("sample", "size"),
        median_total_counts=("total_counts", "median"),
        median_n_genes=("n_genes_by_counts", "median")
    )
)

# Normalize each cell to a common target library size
sc.pp.normalize_total(
    adata,
    target_sum=1e4
)

# Log-transform
sc.pp.log1p(adata)

# Preserve the complete normalized/log-transformed expression matrix
adata.raw = adata

print("Normalization and log transformation complete.")
print("Target sum per cell: 10,000")
print("Full normalized/log-transformed matrix stored in adata.raw.")


Normalization and log transformation complete.
Target sum per cell: 10,000
Full normalized/log-transformed matrix stored in adata.raw.


In [5]:
normalization_summary["target_sum"] = 10_000

normalization_summary.to_csv(
    preprocessing_dir / "normalization_summary.csv",
    index=True
)

display(normalization_summary)
print(
    "Saved:",
    preprocessing_dir / "normalization_summary.csv"
)


,cells,median_total_counts,median_n_genes,target_sum
sample,,,,
HD1,7121,4338.0,2011.0,10000
HD2,10860,4112.0,2041.0,10000
P1,8908,5120.5,2263.0,10000
P2,12292,4078.5,1922.0,10000
P3,11641,4228.0,1968.0,10000
P4,9415,4492.0,1977.0,10000
P5,2277,2616.0,1336.0,10000
P6,8091,3426.0,1677.0,10000
P7,8265,3966.0,1964.0,10000


Saved: /home/naimur-neer/Documents/sc-project3-IFN-I/sc-project3-IFN-I/results/preprocessing_tables/normalization_summary.csv


## 2.5 Identify highly variable genes

Highly variable genes (HVGs) show substantial expression variation across
cells and are informative for downstream dimensionality reduction and
clustering.

The top 2,500 HVGs are selected. `sample` is supplied as the batch key so
that sample-specific effects are considered during HVG selection.


In [6]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2500,
    batch_key="sample"
)

n_hvgs = int(adata.var["highly_variable"].sum())

print("Highly variable genes:", n_hvgs)


Highly variable genes: 2500


## 2.6 HVG diagnostic plot

This plot provides a visual quality check of the highly variable gene
selection.

It is a preprocessing diagnostic rather than a biological result figure.


In [7]:
sc.pl.highly_variable_genes(
    adata,
    show=False
)

hvg_figure = figures_dir / "08_highly_variable_genes.png"

plt.savefig(
    hvg_figure,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

print("Saved:", hvg_figure)


Saved: /home/naimur-neer/Documents/sc-project3-IFN-I/sc-project3-IFN-I/results/figures/08_highly_variable_genes.png


## 2.7 Save HVG summary tables

The summary table records the number of genes available and the number
selected as HVGs.

The selected-gene table provides the gene names and the HVG statistics
calculated by Scanpy.


In [8]:
hvg_summary = pd.DataFrame({
    "metric": [
        "total_genes",
        "selected_hvgs",
        "requested_hvgs"
    ],
    "value": [
        adata.n_vars,
        int(adata.var["highly_variable"].sum()),
        2500
    ]
})

hvg_summary.to_csv(
    preprocessing_dir / "hvg_summary.csv",
    index=False
)

hvg_columns = ["highly_variable"]

for col in [
    "highly_variable_rank",
    "means",
    "dispersions",
    "dispersions_norm",
    "highly_variable_nbatches"
]:
    if col in adata.var.columns:
        hvg_columns.append(col)

top_hvgs = adata.var.loc[
    adata.var["highly_variable"],
    hvg_columns
].copy()

top_hvgs.insert(0, "gene", top_hvgs.index)

top_hvgs.to_csv(
    preprocessing_dir / "top_hvgs.csv",
    index=False
)

print("Saved:", preprocessing_dir / "hvg_summary.csv")
print("Saved:", preprocessing_dir / "top_hvgs.csv")

display(hvg_summary)
display(top_hvgs.head(20))


Saved: /home/naimur-neer/Documents/sc-project3-IFN-I/sc-project3-IFN-I/results/preprocessing_tables/hvg_summary.csv
Saved: /home/naimur-neer/Documents/sc-project3-IFN-I/sc-project3-IFN-I/results/preprocessing_tables/top_hvgs.csv


,metric,value
0,total_genes,36601
1,selected_hvgs,2500
2,requested_hvgs,2500


,gene,highly_variable,means,dispersions,dispersions_norm,highly_variable_nbatches
PRDM16,PRDM16,True,0.004336,1.332661,0.702191,3
AL365255.1,AL365255.1,True,0.025127,1.531308,1.182826,7
CA6,CA6,True,0.087601,1.548953,1.271124,5
KAZN,KAZN,True,0.143987,2.160090,2.934500,10
AL357873.1,AL357873.1,True,0.004630,1.329281,0.693029,3
FHAD1,FHAD1,True,0.032670,1.358273,0.745146,4
PADI4,PADI4,True,0.208372,1.659968,1.421804,8
ALPL,ALPL,True,0.005414,1.685695,1.504421,3
C1QA,C1QA,True,0.033313,1.456875,0.989176,6
C1QB,C1QB,True,0.011035,1.252615,0.702252,4


## 2.8 Subset the dataset to HVGs

The AnnData object is restricted to the selected 2,500 HVGs.

The number of cells does not change. The number of genes decreases to the
selected HVG set.

The full normalized/log-transformed expression matrix remains available
through `adata.raw`.


In [9]:
adata = adata[
    :,
    adata.var["highly_variable"]
].copy()

print("After HVG subsetting:")
print(f"Cells: {adata.n_obs:,}")
print(f"Genes: {adata.n_vars:,}")
print("Full normalized matrix retained in adata.raw:", adata.raw is not None)


After HVG subsetting:
Cells: 87,448
Genes: 2,500
Full normalized matrix retained in adata.raw: True


In [10]:
hvg_subset_file = preprocessing_dir / "hvg_subset.h5ad"

adata.write_h5ad(hvg_subset_file)

print("Saved:", hvg_subset_file)


Saved: /home/naimur-neer/Documents/sc-project3-IFN-I/sc-project3-IFN-I/results/preprocessing_tables/hvg_subset.h5ad


## 2.9 Regress out technical effects

Variation associated with total UMI counts and mitochondrial RNA percentage
is regressed out before scaling and PCA.

This is intended to reduce technical/quality-associated variation that could
otherwise influence downstream dimensionality reduction.


In [11]:
sc.pp.regress_out(
    adata,
    [
        "total_counts",
        "pct_counts_mt"
    ]
)

print("Regression complete.")


/home/naimur-neer/Documents/sc-project3-IFN-I/.venv/lib/python3.11/site-packages/scanpy/preprocessing/_simple.py:666: NumbaPerformanceWarning: '@' is faster on contiguous arrays, called on (Array(float64, 1, 'A', False, aligned=True), Array(float64, 2, 'C', False, aligned=True))
  data[i] -= regressor[i] @ coeff
/home/naimur-neer/Documents/sc-project3-IFN-I/.venv/lib/python3.11/site-packages/scanpy/preprocessing/_simple.py:666: NumbaPerformanceWarning: '@' is faster on contiguous arrays, called on (Array(float64, 1, 'A', False, aligned=True), Array(float64, 2, 'C', False, aligned=True))
  data[i] -= regressor[i] @ coeff


Regression complete.


## 2.10 Scale the expression matrix

Genes are standardized so that they have comparable scales during PCA.

Values are clipped at ±10 to limit the influence of extreme values.


In [12]:
sc.pp.scale(
    adata,
    max_value=10
)

print("Scaling complete.")


Scaling complete.


## 2.11 Save the final Section 2 dataset

This file is the handoff point to the dimensionality-reduction section.

The next section can start from this dataset and perform:

PCA → batch correction → neighbors → UMAP → clustering.


In [13]:
final_file = (
    preprocessing_dir /
    "normalized_hvg_regressed_scaled.h5ad"
)

adata.write_h5ad(final_file)

print("Saved:", final_file)

print("\n==============================")
print("SECTION 2 FINAL DATASET")
print("==============================")
print(f"Cells: {adata.n_obs:,}")
print(f"Genes: {adata.n_vars:,}")
print(f"HVGs represented in adata: {adata.n_vars:,}")


Saved: /home/naimur-neer/Documents/sc-project3-IFN-I/sc-project3-IFN-I/results/preprocessing_tables/normalized_hvg_regressed_scaled.h5ad

SECTION 2 FINAL DATASET
Cells: 87,448
Genes: 2,500
HVGs represented in adata: 2,500


# Section 2 output summary

| Output | Meaning |
|---|---|
| `normalization_summary.csv` | Sample-level record of cell counts and QC metrics entering normalization |
| `08_highly_variable_genes.png` | Diagnostic visualization of HVG selection |
| `hvg_summary.csv` | Number of genes available and number selected as HVGs |
| `top_hvgs.csv` | Selected HVGs and their Scanpy variability statistics |
| `hvg_subset.h5ad` | Normalized/log-transformed dataset restricted to selected HVGs |
| `normalized_hvg_regressed_scaled.h5ad` | Final Section 2 dataset after HVG selection, regression, and scaling |

## Interpretation

Section 1 established which cells and genes should be retained.

Section 2 transforms that trusted expression matrix into a form suitable for
dimensionality reduction and clustering.

The final output is:

`normalized_hvg_regressed_scaled.h5ad`

which becomes the input to Section 3.
